In [ ]:
!pip install transformers
!pip install vaderSentiment
!pip install tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm


In [ ]:
dataset_news = pd.read_csv("news.csv")

In [ ]:
dataset_news.head()

,id,article_language,title,full_text,author,publish_dttm,modified_dttm,url,tags,content_hash,last_updated
0,2,en,OpenAI CEO urges U.S. to prepare for AI ‘super...,OpenAI Chief Executive Sam Altman said U.S. po...,francisco-rodrigues,2026-04-06 15:47:00,2026-04-06 15:47:00,https://www.coindesk.com/tech/2026/04/06/opena...,{artificial-intelligence},82176300df2e50c4c4a8d17085f8ac4d,2026-04-20 22:54:02.084447
1,3,en,Jamie Dimon says JPMorgan must move faster as ...,JPMorgan (JPM) CEO Jamie Dimon said the bank m...,"helene-braun, ai-boost",2026-04-06 15:41:00,2026-04-06 15:41:00,https://www.coindesk.com/markets/2026/04/06/ja...,"{jpm-coin,tokenization,stablecoins}",5773082e8fd1f224f57929c0dfb2cdcf,2026-04-20 22:54:03.114846
2,577,en,CoinDesk 20 performance update: index falls 3....,CoinDesk Indices presents its daily market upd...,coindesk-indices,2026-03-26 13:18:00,2026-03-26 13:18:00,https://www.coindesk.com/coindesk-indices/2026...,"{coindesk-indices,coindesk-20,charts,prices}",e5f1dda47b8c6fec640e3aa8f97f7f08,2026-04-20 22:57:24.217337
3,6,en,Bitmine's ether treasury hits 4.8 million ETH ...,Bitmine Immersion Technologies (BMNR) said it ...,shaurya-malwa,2026-04-06 13:46:00,2026-04-06 13:46:00,https://www.coindesk.com/markets/2026/04/06/bi...,"{digital-asset-treasury,staking}",f445539c0c97f1654444adff9b6ba01c,2026-04-20 22:54:07.308005
4,578,en,MARA Holdings higher by 10% after selling $1.1...,"MARA Holdings (MARA) sold 15,133 bitcoin for a...","james-van-straten, ai-boost",2026-03-26 12:23:00,2026-03-26 12:23:00,https://www.coindesk.com/markets/2026/03/26/ma...,{bitcoin},05a8131f6ecb0b0dfb265b815339a98b,2026-04-20 22:57:25.293738


In [ ]:
df_text = pd.DataFrame(dataset_news)[['publish_dttm', 'full_text']]

In [ ]:
df_text.head()

,publish_dttm,full_text
0,2026-04-06 15:47:00,OpenAI Chief Executive Sam Altman said U.S. po...
1,2026-04-06 15:41:00,JPMorgan (JPM) CEO Jamie Dimon said the bank m...
2,2026-03-26 13:18:00,CoinDesk Indices presents its daily market upd...
3,2026-04-06 13:46:00,Bitmine Immersion Technologies (BMNR) said it ...
4,2026-03-26 12:23:00,"MARA Holdings (MARA) sold 15,133 bitcoin for a..."


In [ ]:
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
contains_urls = df_text['full_text'].str.contains(url_pattern, regex=True)

if contains_urls.any():
    print("There is a URL in the text.")
else:
    print("There are no URLs in the text.")

There is a URL in the text.


In [ ]:
df_text['full_text'] = df_text['full_text'].str.replace(url_pattern, '', regex=True)

In [ ]:
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
contains_urls = df_text['full_text'].str.contains(url_pattern, regex=True)

if contains_urls.any():
    print("There is a URL in the text.")
else:
    print("There are no URLs in the text.")

There are no URLs in the text.


In [ ]:
number_pattern = r'\d+'
contains_numbers = df_text['full_text'].str.contains(number_pattern, regex=True)

if contains_numbers.any():
    print("There are numbers in the text.")
else:
    print("There are no numbers in the text.")

There are numbers in the text.


In [ ]:
df_text['full_text'] = df_text['full_text'].str.replace(r'\d+', '', regex=True)

In [ ]:
number_pattern = r'\d+'
contains_numbers = df_text['full_text'].str.contains(number_pattern, regex=True)

if contains_numbers.any():
    print("There are numbers in the text.")
else:
    print("There are no numbers in the text.")

There are no numbers in the text.


In [ ]:
mention_pattern = r'@\w+'
contains_mentions = df_text['full_text'].str.contains(mention_pattern, regex=True)

if contains_mentions.any():
    print("There are mentions in the text.")
else:
    print("There are no mentions in the text.")

There are mentions in the text.


In [ ]:
df_text['full_text'] = df_text['full_text'].str.replace(mention_pattern, '', regex=True)

In [ ]:
contains_mentions_after = df_text['full_text'].str.contains(mention_pattern, regex=True)
if contains_mentions_after.any():
    print("There are mentions in the text after removal.")
else:
    print("There are no mentions in the text after removal.")

There are no mentions in the text after removal.


In [ ]:
df_text['publish_dttm'] = pd.to_datetime(df_text['publish_dttm'])

print(df_text[:5])

         publish_dttm                                          full_text
0 2026-04-06 15:47:00  OpenAI Chief Executive Sam Altman said U.S. po...
1 2026-04-06 15:41:00  JPMorgan (JPM) CEO Jamie Dimon said the bank m...
2 2026-03-26 13:18:00  CoinDesk Indices presents its daily market upd...
3 2026-04-06 13:46:00  Bitmine Immersion Technologies (BMNR) said it ...
4 2026-03-26 12:23:00  MARA Holdings (MARA) sold , bitcoin for approx...


In [ ]:
analyzer = SentimentIntensityAnalyzer()

df_text['VADER_scores'] = df_text['full_text'].apply(lambda x: analyzer.polarity_scores(x))
df_text['compound'] = df_text['VADER_scores'].apply(lambda d: d['compound'])

In [ ]:
df_text.head()

         publish_dttm                                          full_text  \
0 2026-04-06 15:47:00  OpenAI Chief Executive Sam Altman said U.S. po...   
1 2026-04-06 15:41:00  JPMorgan (JPM) CEO Jamie Dimon said the bank m...   
2 2026-03-26 13:18:00  CoinDesk Indices presents its daily market upd...   
3 2026-04-06 13:46:00  Bitmine Immersion Technologies (BMNR) said it ...   
4 2026-03-26 12:23:00  MARA Holdings (MARA) sold , bitcoin for approx...   

                                        VADER_scores  compound  
0  {'neg': 0.072, 'neu': 0.814, 'pos': 0.114, 'co...    0.9792  
1  {'neg': 0.054, 'neu': 0.863, 'pos': 0.083, 'co...    0.9132  
2  {'neg': 0.022, 'neu': 0.978, 'pos': 0.0, 'comp...   -0.1326  
3  {'neg': 0.018, 'neu': 0.923, 'pos': 0.058, 'co...    0.8294  
4  {'neg': 0.032, 'neu': 0.899, 'pos': 0.068, 'co...    0.7713  


In [ ]:
def categorize_sentiment(compound_score):
    if compound_score > 0.05:
        return 1
    elif compound_score < -0.05:
        return -1
    else:
        return 0

df_text['VADER_Sentiment_Scores'] = df_text['compound'].apply(categorize_sentiment)

In [ ]:
df_text.head(5)

,publish_dttm,full_text,VADER_scores,compound,VADER_Sentiment_Scores
0,2026-04-06 15:47:00,OpenAI Chief Executive Sam Altman said U.S. po...,"{'neg': 0.072, 'neu': 0.814, 'pos': 0.114, 'co...",0.9792,1
1,2026-04-06 15:41:00,JPMorgan (JPM) CEO Jamie Dimon said the bank m...,"{'neg': 0.054, 'neu': 0.863, 'pos': 0.083, 'co...",0.9132,1
2,2026-03-26 13:18:00,CoinDesk Indices presents its daily market upd...,"{'neg': 0.022, 'neu': 0.978, 'pos': 0.0, 'comp...",-0.1326,-1
3,2026-04-06 13:46:00,Bitmine Immersion Technologies (BMNR) said it ...,"{'neg': 0.018, 'neu': 0.923, 'pos': 0.058, 'co...",0.8294,1
4,2026-03-26 12:23:00,"MARA Holdings (MARA) sold , bitcoin for approx...","{'neg': 0.032, 'neu': 0.899, 'pos': 0.068, 'co...",0.7713,1


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from tqdm import tqdm
from torch.nn.functional import softmax

def sentim_analyzer(df, tokenizer, model):
    for i in tqdm(df.index):
        try:
            text_content = df.loc[i, 'full_text']
        except:
            return print(' \'text\' column might be missing from dataframe')
        input = tokenizer(text_content, padding=True, truncation=True, return_tensors='pt')
        output = model(**input)
        predictions = softmax(output.logits, dim=-1)
        df.loc[i, 'Positive'] = predictions[0][0].tolist()
        df.loc[i, 'Negative'] = predictions[0][1].tolist()
        df.loc[i, 'Neutral']  = predictions[0][2].tolist()
    return df

# Use the modified function:
df_text = sentim_analyzer(df_text, tokenizer, model)

100%|██████████| 11375/11375 [2:29:34<00:00,  1.27it/s]


In [ ]:
df_text.head(5)

,publish_dttm,full_text,VADER_scores,compound,VADER_Sentiment_Scores,Positive,Negative,Neutral
0,2026-04-06 15:47:00,OpenAI Chief Executive Sam Altman said U.S. po...,"{'neg': 0.072, 'neu': 0.814, 'pos': 0.114, 'co...",0.9792,1,0.174666,0.032600,0.792735
1,2026-04-06 15:41:00,JPMorgan (JPM) CEO Jamie Dimon said the bank m...,"{'neg': 0.054, 'neu': 0.863, 'pos': 0.083, 'co...",0.9132,1,0.119181,0.015326,0.865494
2,2026-03-26 13:18:00,CoinDesk Indices presents its daily market upd...,"{'neg': 0.022, 'neu': 0.978, 'pos': 0.0, 'comp...",-0.1326,-1,0.027403,0.561865,0.410732
3,2026-04-06 13:46:00,Bitmine Immersion Technologies (BMNR) said it ...,"{'neg': 0.018, 'neu': 0.923, 'pos': 0.058, 'co...",0.8294,1,0.250705,0.019062,0.730234
4,2026-03-26 12:23:00,"MARA Holdings (MARA) sold , bitcoin for approx...","{'neg': 0.032, 'neu': 0.899, 'pos': 0.068, 'co...",0.7713,1,0.885461,0.081486,0.033053


In [ ]:
df_text['BERT_Compound_intermediate'] = df_text['Positive'] - df_text['Negative']

df_text['BERT_Compound'] = np.tanh(df_text['BERT_Compound_intermediate'])

In [ ]:
df_text.head(5)

,publish_dttm,full_text,VADER_scores,compound,VADER_Sentiment_Scores,Positive,Negative,Neutral,BERT_Compound_intermediate,BERT_Compound
0,2026-04-06 15:47:00,OpenAI Chief Executive Sam Altman said U.S. po...,"{'neg': 0.072, 'neu': 0.814, 'pos': 0.114, 'co...",0.9792,1,0.174666,0.032600,0.792735,0.142066,0.141118
1,2026-04-06 15:41:00,JPMorgan (JPM) CEO Jamie Dimon said the bank m...,"{'neg': 0.054, 'neu': 0.863, 'pos': 0.083, 'co...",0.9132,1,0.119181,0.015326,0.865494,0.103855,0.103483
2,2026-03-26 13:18:00,CoinDesk Indices presents its daily market upd...,"{'neg': 0.022, 'neu': 0.978, 'pos': 0.0, 'comp...",-0.1326,-1,0.027403,0.561865,0.410732,-0.534462,-0.488784
3,2026-04-06 13:46:00,Bitmine Immersion Technologies (BMNR) said it ...,"{'neg': 0.018, 'neu': 0.923, 'pos': 0.058, 'co...",0.8294,1,0.250705,0.019062,0.730234,0.231643,0.227587
4,2026-03-26 12:23:00,"MARA Holdings (MARA) sold , bitcoin for approx...","{'neg': 0.032, 'neu': 0.899, 'pos': 0.068, 'co...",0.7713,1,0.885461,0.081486,0.033053,0.803974,0.666253


In [ ]:
def categorize_sentiment(compound_score):
    if compound_score > 0.05:
        return 1
    elif compound_score < -0.05:
        return -1
    else:
        return 0

df_text['BERT_Sentiment_Scores'] = df_text['BERT_Compound'].apply(categorize_sentiment)

In [ ]:
df_text.tail(5)

,publish_dttm,full_text,VADER_scores,compound,VADER_Sentiment_Scores,Positive,Negative,Neutral,BERT_Compound_intermediate,BERT_Compound,BERT_Sentiment_Scores
11370,2025-01-02 10:36:00,"Tether's USDT, the world's leading dollar-pegg...","{'neg': 0.051, 'neu': 0.867, 'pos': 0.082, 'co...",0.7612,1,0.007908,0.969232,0.022860,-0.961323,-0.744867,-1
11371,2025-01-02 10:10:00,KuCoin has introduced a feature for merchants ...,"{'neg': 0.022, 'neu': 0.947, 'pos': 0.03, 'com...",0.3612,1,0.200677,0.009394,0.789929,0.191283,0.188983,1
11372,2025-01-02 08:02:00,Crypto majors zoomed higher in the past hours...,"{'neg': 0.028, 'neu': 0.869, 'pos': 0.103, 'co...",0.9890,1,0.944268,0.014745,0.040986,0.929523,0.730371,1
11373,2025-01-02 06:30:00,Traders are no longer chasing upside in Nasdaq...,"{'neg': 0.072, 'neu': 0.874, 'pos': 0.054, 'co...",-0.4926,-1,0.030461,0.948982,0.020557,-0.918521,-0.725197,-1
11374,2025-01-02 06:00:00,Spanish banking giant Banco Bilbao Vizcaya Arg...,"{'neg': 0.008, 'neu': 0.907, 'pos': 0.085, 'co...",0.9916,1,0.474831,0.012363,0.512806,0.462468,0.432093,1


In [ ]:
columns_to_drop = ["full_text", "VADER_scores", "Positive", "Negative", "Neutral", "BERT_Compound_intermediate"]
df_text = df_text.drop(columns=columns_to_drop)

In [ ]:
df_text.head(5)

,publish_dttm,compound,VADER_Sentiment_Scores,BERT_Compound,BERT_Sentiment_Scores
0,2026-04-06 15:47:00,0.9792,1,0.141118,1
1,2026-04-06 15:41:00,0.9132,1,0.103483,1
2,2026-03-26 13:18:00,-0.1326,-1,-0.488784,-1
3,2026-04-06 13:46:00,0.8294,1,0.227587,1
4,2026-03-26 12:23:00,0.7713,1,0.666253,1


In [ ]:
path = "/content/drive/My Drive/MyData/Sentiment_Scores_Dataset.csv"
df_text.to_csv(path, index=False)


'publish_dttm,compound,VADER_Sentiment_Scores,BERT_Compound,BERT_Sentiment_Scores\n2025-10-17 15:23:00,0.8361,1,0.07929322775045734,1\n2025-08-30 13:00:00,0.9911,1,-0.006973291404406426,0\n2025-06-24 19:00:00,0.84,1,0.14660673208642153,1\n2026-04-08 13:14:00,0.1779,1,0.3421540585034841,1\n2025-09-24 16:00:00,0.9888,1,0.05017818910220348,1\n2026-01-12 05:16:00,-0.9713,-1,-0.7316559539289786,-1\n2025-03-01 04:30:00,0.996,1,-0.1438275811218339,-1\n2025-05-15 11:10:00,0.8147,1,0.027230205226223315,0\n2025-12-18 08:05:00,0.9283,1,0.6385720270498899,1\n2026-04-12 14:00:00,-0.9356,-1,-0.39668233153958227,-1\n2026-03-31 16:59:00,-0.0516,-1,0.6619886346570313,1\n2025-06-28 14:33:00,0.9492,1,0.5410178921407728,1\n2025-03-12 11:15:00,0.4178,1,-0.6806536781033987,-1\n2025-08-09 15:42:00,0.8625,1,-0.12306462903583698,-1\n2025-09-08 11:15:00,0.9474,1,-0.3633054901026876,-1\n2025-01-08 19:37:00,0.0284,0,-0.7394243286024453,-1\n2025-07-09 01:29:00,0.9871,1,0.6455840767352411,1\n2025-10-23 09:59:00,0.9

In [ ]:
df_text.to_csv('Sentiment_Scores_Dataset.csv', index=False)


In [ ]:
df_text

,publish_dttm,compound,VADER_Sentiment_Scores,BERT_Compound,BERT_Sentiment_Scores
0,2026-04-06 15:47:00,0.9792,1,0.141118,1
1,2026-04-06 15:41:00,0.9132,1,0.103483,1
2,2026-03-26 13:18:00,-0.1326,-1,-0.488784,-1
3,2026-04-06 13:46:00,0.8294,1,0.227587,1
4,2026-03-26 12:23:00,0.7713,1,0.666253,1
...,...,...,...,...,...
11370,2025-01-02 10:36:00,0.7612,1,-0.744867,-1
11371,2025-01-02 10:10:00,0.3612,1,0.188983,1
11372,2025-01-02 08:02:00,0.9890,1,0.730371,1
11373,2025-01-02 06:30:00,-0.4926,-1,-0.725197,-1


In [ ]:
dataset_bitcointalk

,id,article_language,title,full_text,author,publish_dttm,modified_dttm,url,tags,content_hash,last_updated
0,2,en,OpenAI CEO urges U.S. to prepare for AI ‘super...,OpenAI Chief Executive Sam Altman said U.S. po...,francisco-rodrigues,2026-04-06 15:47:00,2026-04-06 15:47:00,https://www.coindesk.com/tech/2026/04/06/opena...,{artificial-intelligence},82176300df2e50c4c4a8d17085f8ac4d,2026-04-20 22:54:02.084447
1,3,en,Jamie Dimon says JPMorgan must move faster as ...,JPMorgan (JPM) CEO Jamie Dimon said the bank m...,"helene-braun, ai-boost",2026-04-06 15:41:00,2026-04-06 15:41:00,https://www.coindesk.com/markets/2026/04/06/ja...,"{jpm-coin,tokenization,stablecoins}",5773082e8fd1f224f57929c0dfb2cdcf,2026-04-20 22:54:03.114846
2,577,en,CoinDesk 20 performance update: index falls 3....,CoinDesk Indices presents its daily market upd...,coindesk-indices,2026-03-26 13:18:00,2026-03-26 13:18:00,https://www.coindesk.com/coindesk-indices/2026...,"{coindesk-indices,coindesk-20,charts,prices}",e5f1dda47b8c6fec640e3aa8f97f7f08,2026-04-20 22:57:24.217337
3,6,en,Bitmine's ether treasury hits 4.8 million ETH ...,Bitmine Immersion Technologies (BMNR) said it ...,shaurya-malwa,2026-04-06 13:46:00,2026-04-06 13:46:00,https://www.coindesk.com/markets/2026/04/06/bi...,"{digital-asset-treasury,staking}",f445539c0c97f1654444adff9b6ba01c,2026-04-20 22:54:07.308005
4,578,en,MARA Holdings higher by 10% after selling $1.1...,"MARA Holdings (MARA) sold 15,133 bitcoin for a...","james-van-straten, ai-boost",2026-03-26 12:23:00,2026-03-26 12:23:00,https://www.coindesk.com/markets/2026/03/26/ma...,{bitcoin},05a8131f6ecb0b0dfb265b815339a98b,2026-04-20 22:57:25.293738
...,...,...,...,...,...,...,...,...,...,...,...
11370,13217,en,Tether's Market Value Sees Sharpest Decline Si...,"Tether's USDT, the world's leading dollar-pegg...",omkar-godbole,2025-01-02 10:36:00,2025-01-02 10:36:00,https://www.coindesk.com/markets/2025/01/02/te...,"{tether,usdt,stablecoins,markets,bitcoin}",086bac870c390c651b71e84ef44ec125,2026-04-21 20:07:19.263504
11371,13218,en,KuCoin Enables Crypto Point-of-Sale Payments b...,KuCoin has introduced a feature for merchants ...,helene-braun,2025-01-02 10:10:00,2025-01-02 10:23:00,https://www.coindesk.com/business/2025/01/02/k...,"{point-of-sale,payments,kucoin,crypto-exchanges}",68f6170b318cdb9dc5929eaa1dfa80c9,2026-04-21 20:07:21.611766
11372,13219,en,XRP Rockets 11% as Bitcoin Starts New Year Wit...,Crypto majors zoomed higher in the past 24 hou...,shaurya-malwa,2025-01-02 08:02:00,2025-01-02 08:02:00,https://www.coindesk.com/markets/2025/01/02/xr...,"{xrp,bitcoin,markets}",45b14fb116b876f1eb344445012f0557,2026-04-21 20:07:23.474913
11373,13220,en,MicroStrategy's Bullish Call Skew Disappears i...,Traders are no longer chasing upside in Nasdaq...,omkar-godbole,2025-01-02 06:30:00,2025-01-02 08:48:00,https://www.coindesk.com/markets/2025/01/02/bi...,"{bitcoin,microstrategy,mstr,options}",6a64d7d7705430e050a0a1a382ba8f00,2026-04-21 20:07:25.411624


In [ ]:
len(df_bitcointalk) * 0.05

568.75